In [1]:
import os

print("Current folder:", os.getcwd())
print("Files here:")
print(os.listdir())


Current folder: /Users/raghvendratiwari/Downloads
Files here:
['WhatsApp Image 2026-05-06 at 10.15.24 AM.jpeg', 'WhatsApp Image 2026-05-25 at 11.50.17 AM.jpeg', 'VPD_Uni_Bielefeld_ (1).pdf', 'IMG_0339.HEIC', 'IMG_0268.MOV', 'IMG_0293.HEIC', 'Grade confirmation.pdf', 'googlechrome.dmg', 'IMG_0114.HEIC', 'Exercise_1 (1).ipynb', 'python-3.14.4-macos11.pkg', 'WhatsApp Image 2026-05-28 at 09.29.22.jpeg', 'IMG_0180.HEIC', 'NETZWERK NEU A2 KURSBUCH.pdf', 'Visual Studio Code.app', 'IMG_0246.HEIC', 'IMG_0138.HEIC', '01 - Introduction and Organization (1).mp4', 'IMG_0269.MOV', 'IMG_0179.HEIC', 'Anaconda3-2025.12-2-MacOSX-arm64.pkg', 'WhatsApp Image 2026-05-19 at 5.32.38 AM.jpeg', 'IMG_0227.HEIC', 'Bewerbungsformular_Bewerbungsrunde (1).pdf', 'IMG_0598.JPG', 'IMG_0159.HEIC', 'IMG_0323.HEIC', '15992-25 raghvendratiwari823@gmail.com_sig 2.pdf', 'IMG_0289.HEIC', 'XAI_Solution_2.ipynb', 'IMG_0337.MOV', 'IMG_0374.HEIC', 'IMG_0231.HEIC', 'WhatsApp Image 2026-05-17 at 12.36.26 PM.jpeg', 'xai26-exercise-

In [13]:
import os

file_path = "/Users/raghvendratiwari/Downloads/dblp.ttl"

print("File exists:", os.path.exists(file_path))
print("File size (GB):", round(os.path.getsize(file_path) / (1024**3), 2))


File exists: True
File size (GB): 22.58


In [14]:
file_path = "/Users/raghvendratiwari/Downloads/dblp.ttl"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(10):
        print(f.readline().strip())
        

@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix bf: <http://id.loc.gov/ontologies/bibframe/> .
@prefix bibo: <http://purl.org/ontology/bibo/> .
@prefix bibtex: <http://purl.org/net/nknouf/ns/bibtex#> .
@prefix cito: <http://purl.org/spar/cito/> .
@prefix datacite: <http://purl.org/spar/datacite/> .
@prefix dbo: <http://dbpedia.org/ontology/> .


In [15]:
file_path = "/Users/raghvendratiwari/Downloads/dblp.ttl"

count = 0

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if "primaryAffiliation" in line and "paderborn" in line.lower():
            print(line.strip())
            count += 1
        if count == 10:
            break

dblp:primaryAffiliation "University of Paderborn, Center for Educational Research and Teacher Education, Germany" ;
dblp:primaryAffiliation "Universit\u00E4t Paderborn, Germany" ;
dblp:primaryAffiliation "University of Paderborn, Heinz Nixdorf Institut, Germany" ;
dblp:primaryAffiliation "University of Paderborn, Department of Computer Science, Germany" ;
dblp:primaryAffiliation "University of Paderborn, Germany" ;
dblp:primaryAffiliation "Paderborn University, Germany" ;
dblp:primaryAffiliation "Universit\u00E4t Paderborn, Paderborn, Germany" ;
dblp:primaryAffiliation "University of Paderborn, Department of Computer Science" ;
dblp:primaryAffiliation "University of Paderborn, Germany" ;
dblp:primaryAffiliation "Paderborn University, Departmemnt of Computer Science, Germany" ;


In [20]:
import csv
import re

file_path = "/Users/raghvendratiwari/Downloads/dblp.ttl"
output_path = "paderborn_authors_v2.csv"

def parse_block(block_lines):
    """
    Given a list of lines that form one RDF subject block,
    extract the fields we care about.
    Returns a dict, or None if this block is not a Person/Creator.
    """
    # Join all lines into one string for easier searching
    block = " ".join(line.strip() for line in block_lines)

    # Only process Person/Creator blocks
    if "dblp:Person" not in block and "dblp:Creator" not in block:
        return None

    # Extract subject URI — first token in the block, e.g. <https://dblp.org/pid/...>
    uri_match = re.match(r'<([^>]+)>', block)
    if not uri_match:
        return None
    author_uri = uri_match.group(1)

    # Only keep DBLP person URIs (pid/...)
    if "/pid/" not in author_uri:
        return None

    # Extract primaryCreatorName — value in quotes after the property
    name_match = re.search(r'dblp:primaryCreatorName\s+"([^"]+)"', block)
    author_name = name_match.group(1) if name_match else ""

    # Extract ALL primaryAffiliation values (can be multiple)
    primary_affiliations = re.findall(r'dblp:primaryAffiliation\s+"([^"]+)"', block)

    # Extract ALL affiliation values (past/secondary)
    all_affiliations = re.findall(r'dblp:affiliation\s+"([^"]+)"', block)

    # Combine both lists for Paderborn matching
    combined = primary_affiliations + all_affiliations
    paderborn_matches = [a for a in combined if "paderborn" in a.lower()]

    # FILTER: only keep if at least one PRIMARY affiliation contains Paderborn
    primary_paderborn = [a for a in primary_affiliations if "paderborn" in a.lower()]
    if not primary_paderborn:
        return None  # Skip — Paderborn is only a past affiliation

    return {
        "author_uri": author_uri,
        "author_name": author_name,
        "primary_affiliations": " | ".join(primary_affiliations),
        "all_affiliations": " | ".join(all_affiliations),
        "matched_paderborn_affiliations": " | ".join(paderborn_matches),
    }


# --- Main loop ---

results = []
current_block = []  # Lines collected for the current subject block
block_count = 0
matched_count = 0

print("Starting scan... this will take several minutes on a 22GB file.")

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line_num, line in enumerate(f, 1):

        # Progress indicator every 5 million lines
        if line_num % 5_000_000 == 0:
            print(f"  Lines read: {line_num:,} | Blocks processed: {block_count:,} | Matched: {matched_count}")

        stripped = line.strip()

        # Skip empty lines and comment lines
        if not stripped or stripped.startswith("#"):
            continue

        # A new subject block starts when a line begins with "<https://"
        # and we already have content in current_block — flush the old block first
        if stripped.startswith("<https://") and current_block:
            block_count += 1
            record = parse_block(current_block)
            if record:
                results.append(record)
                matched_count += 1
            current_block = []  # Reset for new block

        current_block.append(line)

        # A block ends when a line ends with " ." (period = end of subject in Turtle)
        if stripped.endswith(" .") or stripped == ".":
            block_count += 1
            record = parse_block(current_block)
            if record:
                results.append(record)
                matched_count += 1
            current_block = []  # Reset

# Handle the very last block in the file if it wasn't flushed
if current_block:
    record = parse_block(current_block)
    if record:
        results.append(record)

print(f"\nDone. Total blocks: {block_count:,} | Paderborn authors found: {len(results)}")

# --- Write CSV ---
with open(output_path, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = [
        "author_uri",
        "author_name",
        "primary_affiliations",
        "all_affiliations",
        "matched_paderborn_affiliations",
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"Saved to: {output_path}")

Starting scan... this will take several minutes on a 22GB file.
  Lines read: 5,000,000 | Blocks processed: 102,153 | Matched: 0
  Lines read: 10,000,000 | Blocks processed: 198,631 | Matched: 0
  Lines read: 15,000,000 | Blocks processed: 299,042 | Matched: 0
  Lines read: 20,000,000 | Blocks processed: 392,932 | Matched: 0
  Lines read: 25,000,000 | Blocks processed: 487,533 | Matched: 0
  Lines read: 30,000,000 | Blocks processed: 582,948 | Matched: 0
  Lines read: 35,000,000 | Blocks processed: 682,402 | Matched: 0
  Lines read: 40,000,000 | Blocks processed: 779,152 | Matched: 0
  Lines read: 45,000,000 | Blocks processed: 878,053 | Matched: 0
  Lines read: 50,000,000 | Blocks processed: 973,438 | Matched: 0
  Lines read: 55,000,000 | Blocks processed: 1,071,206 | Matched: 0
  Lines read: 60,000,000 | Blocks processed: 1,167,641 | Matched: 0
  Lines read: 65,000,000 | Blocks processed: 1,263,271 | Matched: 0
  Lines read: 70,000,000 | Blocks processed: 1,360,091 | Matched: 0
  Lin

In [21]:
import pandas as pd

df = pd.read_csv("paderborn_authors_v2.csv")
print(df.shape)
df.head(10)

(198, 5)


,author_uri,author_name,primary_affiliations,all_affiliations,matched_paderborn_affiliations
0,https://dblp.org/pid/52/7395,Ulrich Wechselberger,"University of Paderborn, Center for Educationa...","University of Paderborn, Center for Educationa...","University of Paderborn, Center for Educationa..."
1,https://dblp.org/pid/52/3346-15,Christian Schmidt,"Universit\u00E4t Paderborn, Germany","Universit\u00E4t Paderborn, Germany","Universit\u00E4t Paderborn, Germany | Universi..."
2,https://dblp.org/pid/53/3909,Harald Selke,"University of Paderborn, Heinz Nixdorf Institu...","University of Paderborn, Heinz Nixdorf Institu...","University of Paderborn, Heinz Nixdorf Institu..."
3,https://dblp.org/pid/53/4856,Patrick Briest,"University of Paderborn, Department of Compute...","University of Paderborn, Department of Compute...","University of Paderborn, Department of Compute..."
4,https://dblp.org/pid/96/1876-1,Tobias Schumacher,"University of Paderborn, Germany","University of Paderborn, Germany","University of Paderborn, Germany | University ..."
5,https://dblp.org/pid/32/10911-1,Peter Stadler,"Paderborn University, Germany","Paderborn University, Germany","Paderborn University, Germany | Paderborn Univ..."
6,https://dblp.org/pid/319/8710,Christian Koldewey,"Universit\u00E4t Paderborn, Paderborn, Germany","Universit\u00E4t Paderborn, Paderborn, Germany","Universit\u00E4t Paderborn, Paderborn, Germany..."
7,https://dblp.org/pid/b/StBottcher,Stefan B\u00F6ttcher,"University of Paderborn, Department of Compute...","University of Paderborn, Department of Compute...","University of Paderborn, Department of Compute..."
8,https://dblp.org/pid/86/4463-2,Michael Baumann,"University of Paderborn, Germany","University of Paderborn, Germany","University of Paderborn, Germany | University ..."
9,https://dblp.org/pid/132/9687,Yasemin Acar,"Paderborn University, Departmemnt of Computer ...","Paderborn University, Departmemnt of Computer ...","Paderborn University, Departmemnt of Computer ..."


In [34]:
import csv
import re

TTL_FILE    = "/Users/raghvendratiwari/Downloads/dblp.ttl"
AUTHORS_CSV = "paderborn_authors_v2.csv"
PAPERS_OUT  = "papers.csv"
EDGES_OUT   = "author_paper_edges.csv"

# Full file — no limit
paderborn_uris = set()
with open(AUTHORS_CSV, "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        paderborn_uris.add(row["author_uri"])
print(f"Loaded {len(paderborn_uris)} Paderborn author URIs")


def parse_block(block_lines):
    block = " ".join(line.strip() for line in block_lines)

    uri_match = re.match(r'<(https?://[^>]+)>', block)
    if not uri_match:
        return None, {}
    subject_uri = uri_match.group(1)

    block_clean = re.sub(r'\[.*?\]', '', block)

    props = {}

    for prop, val in re.findall(
        r'dblp:(\w+)\s+"([^"]+)"(?:\^\^<[^>]+>)?', block_clean
    ):
        props.setdefault(prop, []).append(val)

    segments = re.split(r'(?=dblp:\w+)', block_clean)
    for segment in segments:
        prop_match = re.match(r'dblp:(\w+)', segment)
        if not prop_match:
            continue
        prop = prop_match.group(1)
        uris = re.findall(r'<(https?://[^>]+)>', segment)
        for uri in uris:
            props.setdefault(prop, []).append(uri)

    for val in re.findall(r'\ba\s+dblp:(\w+)', block_clean):
        props.setdefault("rdf_type", []).append(val)

    return subject_uri, props


def is_block_start(s):
    return s.startswith("<https://dblp.org/") and s.endswith(">")

def is_block_end(s):
    return s.endswith(" .") or s == "."


papers = {}
author_paper_pairs = []

def process_block(block_lines):
    if not block_lines:
        return
    subject_uri, props = parse_block(block_lines)
    if not subject_uri or "/rec/" not in subject_uri:
        return

    authored_by     = props.get("authoredBy", [])
    matched_authors = [u for u in authored_by if u in paderborn_uris]
    if not matched_authors:
        return

    papers[subject_uri] = {
        "paper_uri":         subject_uri,
        "title":             props.get("title",             [""])[0],
        "year":              props.get("yearOfPublication", [""])[0],
        "venue_text":        props.get("publishedIn",       [""])[0],
        "venue_uri":         props.get("publishedInStream", [""])[0],
        "rdf_types":         " | ".join(props.get("rdf_type", [])),
        "total_authors":     len(authored_by),
        "paderborn_authors": " | ".join(matched_authors),
    }
    for author_uri in matched_authors:
        author_paper_pairs.append((author_uri, subject_uri))


current_block = []
block_count   = 0

print("Running full file scan — this will take 30–60 minutes...\n")

with open(TTL_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line_num, line in enumerate(f, 1):

        if line_num % 10_000_000 == 0:
            print(f"  Lines: {line_num:,} | Blocks: {block_count:,} | "
                  f"Papers: {len(papers):,} | Edges: {len(author_paper_pairs):,}")

        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        if is_block_start(stripped) and current_block:
            block_count += 1
            process_block(current_block)
            current_block = []

        current_block.append(line)

        if is_block_end(stripped):
            block_count += 1
            process_block(current_block)
            current_block = []

if current_block:
    process_block(current_block)

print(f"\nScan complete.")
print(f"  Blocks processed : {block_count:,}")
print(f"  Papers found     : {len(papers):,}")
print(f"  Edges found      : {len(author_paper_pairs):,}")

# Save
with open(PAPERS_OUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "paper_uri","title","year","venue_text",
        "venue_uri","rdf_types","total_authors","paderborn_authors"
    ])
    writer.writeheader()
    writer.writerows(papers.values())

with open(EDGES_OUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["author_uri", "paper_uri"])
    writer.writerows(author_paper_pairs)

print(f"\nSaved: {PAPERS_OUT}")
print(f"Saved: {EDGES_OUT}")

Loaded 198 Paderborn author URIs
Running full file scan — this will take 30–60 minutes...

  Lines: 10,000,000 | Blocks: 198,631 | Papers: 174 | Edges: 200
  Lines: 20,000,000 | Blocks: 392,932 | Papers: 368 | Edges: 433
  Lines: 30,000,000 | Blocks: 582,948 | Papers: 550 | Edges: 650
  Lines: 40,000,000 | Blocks: 779,152 | Papers: 821 | Edges: 967
  Lines: 50,000,000 | Blocks: 973,438 | Papers: 1,090 | Edges: 1,286
  Lines: 60,000,000 | Blocks: 1,167,641 | Papers: 1,305 | Edges: 1,563
  Lines: 70,000,000 | Blocks: 1,360,091 | Papers: 1,445 | Edges: 1,734
  Lines: 80,000,000 | Blocks: 1,559,673 | Papers: 1,692 | Edges: 2,027
  Lines: 90,000,000 | Blocks: 1,754,884 | Papers: 1,928 | Edges: 2,326
  Lines: 100,000,000 | Blocks: 1,948,677 | Papers: 2,074 | Edges: 2,489
  Lines: 110,000,000 | Blocks: 2,136,003 | Papers: 2,220 | Edges: 2,666
  Lines: 120,000,000 | Blocks: 2,328,072 | Papers: 2,385 | Edges: 2,857
  Lines: 130,000,000 | Blocks: 2,524,661 | Papers: 2,631 | Edges: 3,161
  Lines:

In [35]:
import pandas as pd

edges_df = pd.read_csv("author_paper_edges.csv")

# How many papers have more than 1 Paderborn author?
print(edges_df.groupby("paper_uri")["author_uri"].count().value_counts())


author_uri
1    5489
2     886
3     143
4      20
5       2
9       1
6       1
Name: count, dtype: int64


In [31]:
TARGET_URI = "https://dblp.org/pid/m/BurkhardMonien"  # one of the Paderborn author URIs

TTL_FILE = "/Users/raghvendratiwari/Downloads/dblp.ttl"

current_block = []
found = False

with open(TTL_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line_num, line in enumerate(f, 1):
        if line_num > 3_000_000:
            break

        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        # detect new block start
        if stripped.startswith("<https://dblp.org/") and stripped.endswith(">") and current_block:
            # check if previous block was a paper with our author
            block_text = "".join(current_block)
            if "/rec/" in current_block[0] and TARGET_URI in block_text:
                print("=== RAW BLOCK ===")
                print(block_text)
                found = True
                break
            current_block = []

        current_block.append(line)

        if (stripped.endswith(" .") or stripped == "."):
            block_text = "".join(current_block)
            if "/rec/" in current_block[0] and TARGET_URI in block_text:
                print("=== RAW BLOCK ===")
                print(block_text)
                found = True
                break
            current_block = []

if not found:
    print("Block not found in first 3M lines")

=== RAW BLOCK ===
<https://dblp.org/rec/reference/algo/MonienLW08>
	owl:sameAs <https://doi.org/10.1007/978-3-540-76394-9_28>, <http://dx.doi.org/10.1007/978-3-540-76394-9_28> ;
	rdfs:label "Burkhard Monien et al.: Der Alphabeta-Algorithmus f\u00FCr Spielb\u00E4ume: Wie bringe ich meinen Computer zum Schachspielen?. (2008)" ;
	dblp:doi <https://doi.org/10.1007/978-3-540-76394-9_28> ;
	datacite:hasIdentifier [
		datacite:usesIdentifierScheme datacite:dblp-record ;
		litre:hasLiteralValue "reference/algo/MonienLW08" ;
		a datacite:ResourceIdentifier, datacite:Identifier
	], [
		datacite:usesIdentifierScheme datacite:doi ;
		litre:hasLiteralValue "10.1007/978-3-540-76394-9_28" ;
		a datacite:ResourceIdentifier, datacite:Identifier
	] ;
	dblp:title "Der Alphabeta-Algorithmus f\u00FCr Spielb\u00E4ume: Wie bringe ich meinen Computer zum Schachspielen?." ;
	dblp:bibtexType bibtex:Incollection ;
	dblp:authoredBy <https://dblp.org/pid/m/BurkhardMonien>, <https://dblp.org/pid/07/6960>, <https://

In [36]:
import csv
import pandas as pd

# ── Load all files from Steps 1 and 2 ────────────────────────────────────────
authors_df = pd.read_csv("paderborn_authors_v2.csv")
papers_df  = pd.read_csv("papers.csv")
edges_df   = pd.read_csv("author_paper_edges.csv")

print(f"Authors loaded : {len(authors_df)}")
print(f"Papers loaded  : {len(papers_df)}")
print(f"Edges loaded   : {len(edges_df)}")


# ── PART A: Build venues.csv ──────────────────────────────────────────────────
# Currently venue information is buried inside papers.csv as two columns:
#   venue_text : "VLDB"  (human readable name)
#   venue_uri  : "https://dblp.org/stream/..." (machine URI, may be empty)
#
# We want one row per UNIQUE venue.
# If venue_uri exists, use it as the unique key.
# If not, use venue_text as the key.
# We skip papers with no venue information at all.

venues = {}  # key → {venue_uri, venue_name}

for _, row in papers_df.iterrows():
    venue_text = str(row["venue_text"]).strip()
    venue_uri  = str(row["venue_uri"]).strip()

    # Skip if no venue information at all
    if not venue_text or venue_text == "nan":
        continue

    # Choose the best unique key
    key = venue_uri if (venue_uri and venue_uri != "nan") else venue_text

    # Only add if we haven't seen this venue before
    if key not in venues:
        venues[key] = {
            "venue_uri":  key,
            "venue_name": venue_text,
        }

venues_df = pd.DataFrame(list(venues.values()))
venues_df.to_csv("venues.csv", index=False)
print(f"\nUnique venues  : {len(venues_df)}")


# ── PART B: Build nodes.csv ───────────────────────────────────────────────────
# Every author, paper, and venue becomes a NODE with:
#   node_id      : integer starting from 0  (what PyG needs)
#   node_type    : "author" / "paper" / "venue"
#   label        : human readable name      (what GNNExplainer will show)
#   original_uri : the DBLP URI             (for tracing back to source)
#
# WHY do we need label?
#   So that instead of "node 42 connects to node 891"
#   GNNExplainer can say "Stefan Heindorf authored Paper XYZ"

nodes = []

# Add author nodes
for _, row in authors_df.iterrows():
    nodes.append({
        "node_type":    "author",
        "label":        str(row["author_name"]),
        "original_uri": str(row["author_uri"]),
    })

# Add paper nodes
for _, row in papers_df.iterrows():
    # Use title as label if available, otherwise use URI
    title = str(row["title"]).strip()
    label = title if (title and title != "nan") else str(row["paper_uri"])
    nodes.append({
        "node_type":    "paper",
        "label":        label,
        "original_uri": str(row["paper_uri"]),
    })

# Add venue nodes
for _, row in venues_df.iterrows():
    nodes.append({
        "node_type":    "venue",
        "label":        str(row["venue_name"]),
        "original_uri": str(row["venue_uri"]),
    })

# Convert to DataFrame and assign sequential integer IDs
nodes_df = pd.DataFrame(nodes).reset_index(drop=True)
nodes_df.insert(0, "node_id", nodes_df.index)  # node_id = row number = 0,1,2,3...

nodes_df.to_csv("nodes.csv", index=False)

print(f"\nTotal nodes    : {len(nodes_df)}")
print(nodes_df["node_type"].value_counts().to_string())

# Build a fast lookup: original_uri → node_id
# This is the same as nodes_dict in the AIFB example
uri_to_id = dict(zip(nodes_df["original_uri"], nodes_df["node_id"]))


# ── PART C: Build edges.csv ───────────────────────────────────────────────────
# Two relation types:
#   author_of    : author node → paper node
#   published_in : paper node  → venue node
#
# Each edge is stored as:
#   source_id | relation_type | target_id
# All as integers — ready for PyG

graph_edges = []

# author_of edges — from author_paper_edges.csv
for _, row in edges_df.iterrows():
    src = uri_to_id.get(str(row["author_uri"]))
    dst = uri_to_id.get(str(row["paper_uri"]))

    # Skip if either node is missing (shouldn't happen, but safe to check)
    if src is None or dst is None:
        continue

    graph_edges.append({
        "source_id":     src,
        "relation_type": "author_of",
        "target_id":     dst,
    })

# published_in edges — from papers.csv venue columns
for _, row in papers_df.iterrows():
    venue_text = str(row["venue_text"]).strip()
    venue_uri  = str(row["venue_uri"]).strip()

    if not venue_text or venue_text == "nan":
        continue

    key = venue_uri if (venue_uri and venue_uri != "nan") else venue_text

    src = uri_to_id.get(str(row["paper_uri"]))
    dst = uri_to_id.get(key)

    if src is None or dst is None:
        continue

    graph_edges.append({
        "source_id":     src,
        "relation_type": "published_in",
        "target_id":     dst,
    })

graph_edges_df = pd.DataFrame(graph_edges)
graph_edges_df.to_csv("edges.csv", index=False)

print(f"\nTotal edges    : {len(graph_edges_df)}")
print(graph_edges_df["relation_type"].value_counts().to_string())


# ── PART D: Verify everything is consistent ───────────────────────────────────
print("\n=== Sanity Checks ===")

# Check 1: all author_of source nodes should be type "author"
author_of_edges = graph_edges_df[graph_edges_df["relation_type"] == "author_of"]
src_types = nodes_df.set_index("node_id").loc[author_of_edges["source_id"], "node_type"]
print(f"author_of source types (all should be 'author') : {src_types.value_counts().to_dict()}")

# Check 2: all author_of target nodes should be type "paper"
dst_types = nodes_df.set_index("node_id").loc[author_of_edges["target_id"], "node_type"]
print(f"author_of target types (all should be 'paper')  : {dst_types.value_counts().to_dict()}")

# Check 3: all published_in target nodes should be type "venue"
pub_edges = graph_edges_df[graph_edges_df["relation_type"] == "published_in"]
dst_types2 = nodes_df.set_index("node_id").loc[pub_edges["target_id"], "node_type"]
print(f"published_in target types (all should be 'venue'): {dst_types2.value_counts().to_dict()}")

# Check 4: show one example of each relation in human-readable form
print("\n=== Sample edges (human readable) ===")
id_to_label = dict(zip(nodes_df["node_id"], nodes_df["label"]))
id_to_type  = dict(zip(nodes_df["node_id"], nodes_df["node_type"]))

print("\nauthor_of examples:")
for _, row in author_of_edges.head(3).iterrows():
    print(f"  [{id_to_type[row.source_id]}] {id_to_label[row.source_id]}")
    print(f"    --author_of-->")
    print(f"  [{id_to_type[row.target_id]}] {id_to_label[row.target_id]}")

print("\npublished_in examples:")
for _, row in pub_edges.head(3).iterrows():
    print(f"  [{id_to_type[row.source_id]}] {id_to_label[row.source_id][:60]}")
    print(f"    --published_in-->")
    print(f"  [{id_to_type[row.target_id]}] {id_to_label[row.target_id]}")

Authors loaded : 198
Papers loaded  : 6542
Edges loaded   : 7795

Unique venues  : 1445

Total nodes    : 8185
node_type
paper     6542
venue     1445
author     198

Total edges    : 14180
relation_type
author_of       7795
published_in    6385

=== Sanity Checks ===
author_of source types (all should be 'author') : {'author': 7795}
author_of target types (all should be 'paper')  : {'paper': 7795}
published_in target types (all should be 'venue'): {'venue': 6385}

=== Sample edges (human readable) ===

author_of examples:
  [author] Friedhelm Meyer auf der Heide
    --author_of-->
  [paper] Bin Packing oder \
  [author] Burkhard Monien
    --author_of-->
  [paper] Der Alphabeta-Algorithmus f\u00FCr Spielb\u00E4ume: Wie bringe ich meinen Computer zum Schachspielen?.
  [author] Daniel Warner
    --author_of-->
  [paper] Der Alphabeta-Algorithmus f\u00FCr Spielb\u00E4ume: Wie bringe ich meinen Computer zum Schachspielen?.

published_in examples:
  [paper] Bin Packing oder \
    --publish

In [39]:
import pandas as pd
import re

nodes_df = pd.read_csv("nodes.csv")

def fix_unicode(text):
    if not isinstance(text, str):
        return text
    # Only replace \uXXXX patterns, leave everything else untouched
    return re.sub(
        r'\\u([0-9a-fA-F]{4})',
        lambda m: chr(int(m.group(1), 16)),
        text
    )

nodes_df["label"] = nodes_df["label"].apply(fix_unicode)
nodes_df.to_csv("nodes.csv", index=False)

print("Unicode fixed. Sample paper labels:")
print(nodes_df[nodes_df["node_type"] == "paper"]["label"].head(10).to_string())

Unicode fixed. Sample paper labels:
198                                   Bin Packing oder \
199    Der Alphabeta-Algorithmus für Spielbäume: Wie ...
200                             Teilen von Geheimnissen.
201    Broadcasting: Wie verbreite ich schnell Inform...
202    Jamming-Resistant MAC Protocols for Wireless N...
203    Approximation Algorithms for Multilevel Graph ...
204          Overlay Networks for Peer-to-Peer Networks.
205                              Data Science Education.
206                            Artifacts for the Paper \
207              Supplementary Material for the paper: \


In [40]:
import pandas as pd

nodes_df = pd.read_csv("nodes.csv")
papers_with_backslash = nodes_df[
    (nodes_df["node_type"] == "paper") & 
    (nodes_df["label"].str.endswith("\\"))
]
print(f"Papers with trailing backslash: {len(papers_with_backslash)}")
print(papers_with_backslash["label"].to_string())

Papers with trailing backslash: 64
198                                    Bin Packing oder \
206                             Artifacts for the Paper \
207               Supplementary Material for the paper: \
209                             Artifacts for the Paper \
213                             Artifacts for the Paper \
288                                   zur Arbeitsgruppe \
732                                                     \
735                                                     \
756                                                     \
765                                         5. Workshop \
769                                 Dritter Workshop zu \
788                                 Dritter Workshop zu \
802       Workshop und Fachgruppentreffen der FG OOSE - \
855                                                     \
937                                                     \
1112                                                    \
1123                                 

In [41]:
def clean_label(text):
    if not isinstance(text, str):
        return text
    # Remove trailing backslash and whitespace
    return text.rstrip("\\").strip()

nodes_df["label"] = nodes_df["label"].apply(clean_label)
nodes_df.to_csv("nodes.csv", index=False)

print("Cleaned. Affected rows now:")
print(nodes_df[nodes_df["label"].str.endswith("\\", na=False)])
# Should print empty — 0 rows

Cleaned. Affected rows now:
Empty DataFrame
Columns: [node_id, node_type, label, original_uri]
Index: []


In [42]:
import os
print(os.getcwd())

/Users/raghvendratiwari/Downloads
